# Worked example: end-to-end rate limiting with Redis

Run this notebook one cell at a time to follow requests through:

```text
Notebook client -> NGINX -> FastAPI gateway -> Redis bucket decision -> inference service
```

The notebook makes Redis's role visible: it inspects bucket hashes, demonstrates lazy refill, triggers a `429`, and proves that tenants have independent limits.

## 1. Locate the example and define helpers

The HTTP helper uses Python's standard library. The Docker helpers run commands from the folder containing this module's `docker-compose.yml`.

In [1]:
from pathlib import Path
from pprint import pprint
import json
import subprocess
import time
import urllib.error
import urllib.request


def find_project_dir():
    candidates = [
        Path.cwd(),
        Path.cwd() / "system-design" / "02-rate-limiting",
    ]
    for candidate in candidates:
        if (candidate / "docker-compose.yml").exists() and (candidate / "app" / "token_bucket.lua").exists():
            return candidate.resolve()
    raise FileNotFoundError("Run this notebook from the repository root or 02-rate-limiting folder")


PROJECT_DIR = find_project_dir()
BASE_URL = "http://localhost:8081"


def compose(*args, check=True):
    result = subprocess.run(
        ["docker", "compose", *args],
        cwd=PROJECT_DIR,
        text=True,
        capture_output=True,
        check=check,
    )
    print(result.stdout or result.stderr)
    return result


def redis_cli(*args):
    result = subprocess.run(
        ["docker", "compose", "exec", "-T", "redis", "redis-cli", "--raw", *args],
        cwd=PROJECT_DIR,
        text=True,
        capture_output=True,
        check=True,
    )
    return result.stdout.strip()


def api_request(method, path, *, api_key=None, body=None):
    headers = {"Content-Type": "application/json"}
    if api_key:
        headers["X-API-Key"] = api_key
    data = json.dumps(body).encode() if body is not None else None
    request = urllib.request.Request(BASE_URL + path, data=data, headers=headers, method=method)

    try:
        response = urllib.request.urlopen(request, timeout=10)
    except urllib.error.HTTPError as error:
        response = error

    raw_body = response.read().decode()
    result = {
        "status": response.status,
        "headers": {key.lower(): value for key, value in response.headers.items()},
        "body": json.loads(raw_body) if raw_body else None,
    }
    pprint(result)
    return result


def bucket(key):
    values = redis_cli("HGETALL", key).splitlines()
    state = dict(zip(values[::2], values[1::2])) if values else {}
    return {"key": key, "state": state, "ttl_seconds": redis_cli("TTL", key)}


print(f"Using example at: {PROJECT_DIR}")

Using example at: /Users/Minh/repos/ai-engineering/system-design/02-rate-limiting


## 2. Start the complete stack

Only NGINX publishes `localhost:8081`. FastAPI, Redis, and the mock inference service remain private inside Docker's network.

In [2]:
compose("up", "--build", "-d")
compose("ps")

#1 [02-rate-limiting_gateway internal] load build definition from Dockerfile
#1 transferring dockerfile: 69B done
#1 DONE 0.0s

#2 [02-rate-limiting_inference-service internal] load build definition from Dockerfile
#2 transferring dockerfile: 69B done
#2 DONE 0.0s

#3 [02-rate-limiting_inference-service internal] load .dockerignore
#3 transferring context:
#3 transferring context: 2B done
#3 DONE 0.0s

#4 [02-rate-limiting_gateway internal] load .dockerignore
#4 transferring context: 2B done
#4 DONE 0.0s

#5 [02-rate-limiting_gateway internal] load metadata for docker.io/library/python:3.12-slim
#5 DONE 0.9s

#6 [02-rate-limiting_gateway internal] load build context
#6 transferring context: 363B done
#6 DONE 0.0s

#7 [02-rate-limiting_gateway 4/5] RUN pip install --no-cache-dir -r requirements.txt
#7 CACHED

#8 [02-rate-limiting_gateway 3/5] COPY requirements.txt ./
#8 CACHED

#9 [02-rate-limiting_gateway 2/5] WORKDIR /app
#9 CACHED

#10 [02-rate-limiting_gateway 5/5] COPY app ./app
#1

CompletedProcess(args=['docker', 'compose', 'ps'], returncode=0, stdout='NAME                                   COMMAND                  SERVICE             STATUS              PORTS\n02-rate-limiting-gateway-1             "uvicorn app.gateway…"   gateway             running (healthy)   8080/tcp\n02-rate-limiting-inference-service-1   "uvicorn app.inferen…"   inference-service   running             8002/tcp\n02-rate-limiting-nginx-1               "/docker-entrypoint.…"   nginx               running             80/tcp, 0.0.0.0:8081->8081/tcp\n02-rate-limiting-redis-1               "docker-entrypoint.s…"   redis               running (healthy)   6379/tcp\n', stderr='')

## 3. Start with empty Redis

`FLUSHDB` resets this learning environment. Before any tenant request arrives, Redis holds no rate-limit buckets.

In [3]:
print(redis_cli("FLUSHDB"))
print("Rate-limit keys:", redis_cli("SCAN", "0", "MATCH", "rate_limit:*"))

OK
Rate-limit keys: 0


## 4. Send one request through every component

The local policy gives each tenant:

- A request bucket with capacity `5`, refill rate `1/second`, and cost `1` per request.
- A token bucket with capacity `100`, refill rate `20/second`, and cost approximately `ceil(prompt characters / 4) + max_tokens`.

The gateway maps `dev-key` to the stable identity `tenant-dev`, estimates cost, and asks Redis to atomically check both buckets. Inference runs only when both have capacity.

In [4]:
first = api_request(
    "POST",
    "/v1/chat/completions",
    api_key="dev-key",
    body={"message": "Explain why model tokens cost money.", "max_tokens": 60},
)

assert first["status"] == 200
print("Estimated token cost:", first["body"]["estimated_token_cost"])
print("Remaining request units:", first["headers"]["x-ratelimit-request-remaining"])
print("Remaining token units:", first["headers"]["x-ratelimit-token-remaining"])

{'body': {'data': {'echo': 'Explain why model tokens cost money.',
                   'max_tokens': 60,
                   'message': 'The request reached inference because both '
                              'tenant buckets had capacity.',
                   'model': 'toy-model'},
          'estimated_token_cost': 69,
          'object': 'chat.completion',
          'tenant': 'tenant-dev'},
 'headers': {'connection': 'close',
             'content-length': '251',
             'content-type': 'application/json',
             'date': 'Tue, 09 Jun 2026 07:11:22 GMT',
             'server': 'nginx/1.28.3',
             'x-ratelimit-request-remaining': '4',
             'x-ratelimit-token-remaining': '31'},
 'status': 200}
Estimated token cost: 69
Remaining request units: 4
Remaining token units: 31


## 5. Inspect what Redis actually stores

Redis stores one small hash for each enforced bucket. It does **not** store one historical row per request.

- `tokens` is the remaining capacity at the last successful update.
- `updated_at` is Redis server time at that update.
- The TTL removes inactive buckets after enough time has passed for them to refill.

In [5]:
request_key = "rate_limit:tenant:tenant-dev:requests"
token_key = "rate_limit:tenant:tenant-dev:tokens"

print("Keys currently in Redis:")
print(redis_cli("SCAN", "0", "MATCH", "rate_limit:*").replace("0\n", ""))
print("\nRequest bucket:")
pprint(bucket(request_key))
print("\nToken bucket:")
pprint(bucket(token_key))

Keys currently in Redis:
rate_limit:tenant:tenant-dev:tokens
rate_limit:tenant:tenant-dev:requests

Request bucket:
{'key': 'rate_limit:tenant:tenant-dev:requests',
 'state': {'tokens': '4', 'updated_at': '1780989082.926315'},
 'ttl_seconds': '9'}

Token bucket:
{'key': 'rate_limit:tenant:tenant-dev:tokens',
 'state': {'tokens': '31', 'updated_at': '1780989082.926315'},
 'ttl_seconds': '9'}


## 6. Observe lazy replenishment

There is no background worker incrementing every bucket each second. While we wait, the stored hashes remain unchanged. The next request calculates replenishment using:

```text
available = min(capacity, stored_tokens + elapsed_seconds * refill_rate)
```

It then deducts the new request and writes the resulting state.

In [6]:
before_wait = bucket(token_key)
print("Stored token bucket before waiting:")
pprint(before_wait)

time.sleep(2)

after_wait = bucket(token_key)
print("\nStored token bucket after waiting two seconds, before another request:")
pprint(after_wait)

assert before_wait["state"] == after_wait["state"]
print("\nThe stored values did not change because refill is calculated lazily.")

Stored token bucket before waiting:
{'key': 'rate_limit:tenant:tenant-dev:tokens',
 'state': {'tokens': '31', 'updated_at': '1780989082.926315'},
 'ttl_seconds': '9'}

Stored token bucket after waiting two seconds, before another request:
{'key': 'rate_limit:tenant:tenant-dev:tokens',
 'state': {'tokens': '31', 'updated_at': '1780989082.926315'},
 'ttl_seconds': '6'}

The stored values did not change because refill is calculated lazily.


In [7]:
second = api_request(
    "POST",
    "/v1/chat/completions",
    api_key="dev-key",
    body={"message": "small request", "max_tokens": 5},
)

assert second["status"] == 200
print("\nToken bucket after lazy refill and the second deduction:")
pprint(bucket(token_key))

{'body': {'data': {'echo': 'small request',
                   'max_tokens': 5,
                   'message': 'The request reached inference because both '
                              'tenant buckets had capacity.',
                   'model': 'toy-model'},
          'estimated_token_cost': 9,
          'object': 'chat.completion',
          'tenant': 'tenant-dev'},
 'headers': {'connection': 'close',
             'content-length': '226',
             'content-type': 'application/json',
             'date': 'Tue, 09 Jun 2026 07:11:26 GMT',
             'server': 'nginx/1.28.3',
             'x-ratelimit-request-remaining': '4',
             'x-ratelimit-token-remaining': '91'},
 'status': 200}

Token bucket after lazy refill and the second deduction:
{'key': 'rate_limit:tenant:tenant-dev:tokens',
 'state': {'tokens': '91', 'updated_at': '1780989086.513825'},
 'ttl_seconds': '10'}


## 7. Exhaust the request bucket and receive `429`

Reset Redis, then send six inexpensive requests quickly. The first five consume the request bucket's burst capacity. The sixth is rejected before inference and includes `Retry-After` plus the binding limit type.

In [8]:
print(redis_cli("FLUSHDB"))

burst = []
for number in range(1, 7):
    result = api_request(
        "POST",
        "/v1/chat/completions",
        api_key="dev-key",
        body={"message": "small request", "max_tokens": 5},
    )
    burst.append((number, result["status"], result["headers"].get("x-ratelimit-limit-type")))

print("\nBurst summary:")
pprint(burst)
assert burst[-1][1] == 429

print("\nStored request bucket after rejection:")
pprint(bucket(request_key))

OK
{'body': {'data': {'echo': 'small request',
                   'max_tokens': 5,
                   'message': 'The request reached inference because both '
                              'tenant buckets had capacity.',
                   'model': 'toy-model'},
          'estimated_token_cost': 9,
          'object': 'chat.completion',
          'tenant': 'tenant-dev'},
 'headers': {'connection': 'close',
             'content-length': '226',
             'content-type': 'application/json',
             'date': 'Tue, 09 Jun 2026 07:11:27 GMT',
             'server': 'nginx/1.28.3',
             'x-ratelimit-request-remaining': '4',
             'x-ratelimit-token-remaining': '91'},
 'status': 200}
{'body': {'data': {'echo': 'small request',
                   'max_tokens': 5,
                   'message': 'The request reached inference because both '
                              'tenant buckets had capacity.',
                   'model': 'toy-model'},
          'estimated_token_cost'

The Lua script calculates both bucket states first and commits deductions only when **both** buckets allow the request. A rejected request therefore does not partially charge one bucket while another bucket blocks it.

## 8. Prove tenant isolation

`dev-key` maps to `tenant-dev`, while `team-key` maps to `tenant-team`. Even though `tenant-dev` exhausted its request burst, `tenant-team` starts with independent buckets.

In [9]:
team = api_request(
    "POST",
    "/v1/chat/completions",
    api_key="team-key",
    body={"message": "I have separate tenant limits.", "max_tokens": 10},
)

assert team["status"] == 200
print("\nAll tenant bucket keys:")
print(redis_cli("SCAN", "0", "MATCH", "rate_limit:tenant:*").replace("0\n", ""))

{'body': {'data': {'echo': 'I have separate tenant limits.',
                   'max_tokens': 10,
                   'message': 'The request reached inference because both '
                              'tenant buckets had capacity.',
                   'model': 'toy-model'},
          'estimated_token_cost': 18,
          'object': 'chat.completion',
          'tenant': 'tenant-team'},
 'headers': {'connection': 'close',
             'content-length': '246',
             'content-type': 'application/json',
             'date': 'Tue, 09 Jun 2026 07:11:27 GMT',
             'server': 'nginx/1.28.3',
             'x-ratelimit-request-remaining': '4',
             'x-ratelimit-token-remaining': '82'},
 'status': 200}

All tenant bucket keys:
rate_limit:tenant:tenant-dev:tokens
rate_limit:tenant:tenant-team:tokens
rate_limit:tenant:tenant-team:requests
rate_limit:tenant:tenant-dev:requests


## 9. Reject a request that can never fit

A request estimated above the full token-bucket capacity can never become admissible by waiting. The gateway returns `422` before creating or changing Redis bucket state.

In [10]:
oversized = api_request(
    "POST",
    "/v1/chat/completions",
    api_key="team-key",
    body={
        "message": "This prompt plus its reserved output is larger than the complete bucket capacity.",
        "max_tokens": 100,
    },
)

assert oversized["status"] == 422

{'body': {'detail': {'estimated_token_cost': 121,
                     'maximum_token_cost': 100,
                     'message': 'Estimated request cost exceeds the '
                                'per-request token capacity'}},
 'headers': {'connection': 'close',
             'content-length': '138',
             'content-type': 'application/json',
             'date': 'Tue, 09 Jun 2026 07:11:27 GMT',
             'server': 'nginx/1.28.3'},
 'status': 422}


## 10. Redis state is not usage history

The hashes inspected above answer: **may this request proceed right now?** They expire and do not preserve request-by-request history.

A production system should separately emit durable usage events containing fields such as request ID, tenant, user, model, endpoint, allowed/rejected decision, estimated tokens, actual tokens, and timestamp. Those events belong in PostgreSQL, ClickHouse, a warehouse, or an event pipeline.

## 11. Inspect recent service logs

The gateway logs show accepted and rejected public requests. The inference-service logs show only requests that passed both Redis buckets.

In [11]:
compose("logs", "--tail=40", "nginx", "gateway", "redis", "inference-service")

02-rate-limiting-redis-1  | 1:C 09 Jun 2026 07:11:12.690 * oO0OoO0OoO0Oo Redis is starting oO0OoO0OoO0Oo
02-rate-limiting-redis-1  | 1:C 09 Jun 2026 07:11:12.690 * Redis version=7.4.9, bits=64, commit=00000000, modified=0, pid=1, just started
02-rate-limiting-redis-1  | 1:C 09 Jun 2026 07:11:12.690 # Warning: no config file specified, using the default config. In order to specify a config file use redis-server /path/to/redis.conf
02-rate-limiting-redis-1  | 1:M 09 Jun 2026 07:11:12.690 * monotonic clock: POSIX clock_gettime
02-rate-limiting-redis-1  | 1:M 09 Jun 2026 07:11:12.691 * Running mode=standalone, port=6379.
02-rate-limiting-redis-1  | 1:M 09 Jun 2026 07:11:12.692 * Server initialized
02-rate-limiting-redis-1  | 1:M 09 Jun 2026 07:11:12.693 * Ready to accept connections tcp
02-rate-limiting-gateway-1  | INFO:     Started server process [1]
02-rate-limiting-gateway-1  | INFO:     Waiting for application startup.
02-rate-limiting-gateway-1  | INFO:     Application startup comple

CompletedProcess(args=['docker', 'compose', 'logs', '--tail=40', 'nginx', 'gateway', 'redis', 'inference-service'], returncode=0, stdout='02-rate-limiting-redis-1  | 1:C 09 Jun 2026 07:11:12.690 * oO0OoO0OoO0Oo Redis is starting oO0OoO0OoO0Oo\n02-rate-limiting-redis-1  | 1:C 09 Jun 2026 07:11:12.690 * Redis version=7.4.9, bits=64, commit=00000000, modified=0, pid=1, just started\n02-rate-limiting-redis-1  | 1:C 09 Jun 2026 07:11:12.690 # Warning: no config file specified, using the default config. In order to specify a config file use redis-server /path/to/redis.conf\n02-rate-limiting-redis-1  | 1:M 09 Jun 2026 07:11:12.690 * monotonic clock: POSIX clock_gettime\n02-rate-limiting-redis-1  | 1:M 09 Jun 2026 07:11:12.691 * Running mode=standalone, port=6379.\n02-rate-limiting-redis-1  | 1:M 09 Jun 2026 07:11:12.692 * Server initialized\n02-rate-limiting-redis-1  | 1:M 09 Jun 2026 07:11:12.693 * Ready to accept connections tcp\n02-rate-limiting-gateway-1  | INFO:     Started server proces

## 12. Stop the stack when finished

Run this cell when you no longer need the example containers.

In [12]:
# Uncomment when you want to stop and remove the containers.
# compose("down")